<a href="https://colab.research.google.com/github/dorvarsul/Deep-Learning-OpenUniversity/blob/main/DL_MMN11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Maman 11 — Deep Learning

**Name:** Dor Varsulker  
**ID:** 209299924

## Imports

In [1]:
import torch

##Q1.A

## Broadcasting rules

Shapes are compared from the **right**. A tensor with fewer dimensions is treated as if it had extra size-1 dimensions on the left. Each pair of sizes must be equal, or one of them must be 1 (which is then stretched to the other size, including to 0). Otherwise the shapes are not broadcastable

In [2]:
def my_expand_as(A, B):
  # Rule 1 : A can't have more dimensions than B
  if A.dim() > B.dim():
    raise ValueError(f"A has {A.dim()} dimensions, B has {B.dim()}")

  # Rule 2: compare dimensions from the right, Each of A's sizes must
  # (i) equal B's size in that dimension, or
  # (ii) be 1, so it can be repeated
  for i in range(-1, -A.dim()-1, -1):
    if A.shape[i] != B.shape[i] and A.shape[i] != 1:
      raise ValueError(
          f"Dimension {i} (from right): A has size {A.shape[i]}, B has size {B.shape[i]}"
          )

  # Padding: add size-1 dimensions to A's left untill it has as many dimensions as B
  C = A
  for i in range(B.dim() - A.dim()):
    # Add new dimension to the left e.g, tensor with shape(x) -> shape(1,x)
    C = C.unsqueeze(0)

  # Stretching: C and B now have the same number of dimensions
  # Wherever their sizes differ, the checks above guarantee C's size is 1,
  # so we add B.shape[i] copies along dimensions i to get exactly B's size
  for i in range(C.dim()):
    if B.shape[i] != C.shape[i]:
      C = torch.cat([C] * B.shape[i], dim=i)

  return C


##Q1.B

In [3]:
def my_is_broadcastable(A, B):
  # The result has as many dimensions as the larger tensor
  n = max(A.dim(), B.dim())
  shape = []

  # Compare dimensions from the right; a missing dimension counts as size 1
  for i in range(-1, -n-1, -1):
    a = A.shape[i] if -i <= A.dim() else 1
    b = B.shape[i] if -i <= B.dim() else 1

    if a == b or b == 1:     # equal, or B's 1 is stretched to a
      shape.insert(0, a)
    elif a == 1:             # A's 1 is stretched to b
      shape.insert(0, b)
    else:                    # conflict: neither equal nor 1
      return (False, None)

  return (True, torch.Size(shape))

##Q1.C

In [4]:
def my_broadcast_tensors(A, B):
  # Part B: find the common shape (or fail)
  ok, shape = my_is_broadcastable(A, B)
  if not ok:
    raise ValueError(f"Shapes {A.shape} and {B.shape} are not broadcastable")

  # Part A expects a target tensor, so we create an uninitialized one with the common shape
  target = torch.empty(shape)

  # Expand each input to the common shape using part A
  return my_expand_as(A, target), my_expand_as(B, target)

## Part D — Comparison with PyTorch

A test **passes** if both implementations raise an error, or both succeed with identical shape, dtype and values. Tensors are filled with random integers so equality is exact.

In [5]:
def outcome(fn):
  # Run fn and return ("ok", result) or ("error", None)
  try:
    return ("ok", fn())
  except (RuntimeError, ValueError):   # PyTorch raises RuntimeError, we raise ValueError
    return ("error", None)


def same(x, y):
  return x.shape == y.shape and x.dtype == y.dtype and torch.equal(x, y)


def agree(mine, ref):
  # Both failed, or both succeeded with identical tensor(s)
  if mine[0] != ref[0]:
    return False
  if mine[0] == "error":
    return True
  if isinstance(mine[1], torch.Tensor):
    return same(mine[1], ref[1])
  return all(same(m, r) for m, r in zip(mine[1], ref[1]))


def rand_tensor(shape):
  return torch.randint(-50, 50, size=shape)

### D.1 — Part A, hand-picked cases

7 legal cases, then 5 that must fail.

In [6]:
EXPAND_CASES = [
  ((3,),      (3,)),        # identical
  ((),        (2, 3)),      # scalar
  ((3,),      (2, 3)),      # prepend a dimension
  ((2, 1),    (2, 4)),      # stretch the last dimension
  ((1, 3, 1), (2, 3, 4)),   # prepend + stretch several dimensions
  ((1,),      (0,)),        # stretch 1 -> 0
  ((0,),      (2, 0)),      # keep a size-0 dimension
  ((2, 3),    (3,)),        # error: A has more dimensions
  ((3,),      (4,)),        # error: size conflict
  ((3,),      (3, 1)),      # error: one-directional
  ((2, 1),    (1, 2)),      # error: one-directional
  ((0,),      (1,)),        # error: 0 can't become 1
]


def test_expand(cases):
  passed = 0
  for a_shape, b_shape in cases:
    A, B = rand_tensor(a_shape), rand_tensor(b_shape)
    ref = outcome(lambda: A.expand_as(B))
    ok = agree(outcome(lambda: my_expand_as(A, B)), ref)
    passed += ok
    print(f"{str(a_shape):<10} -> {str(b_shape):<10} ref: {ref[0]:<6} {'PASS' if ok else 'FAIL'}")
  return passed

In [7]:
a_passed = test_expand(EXPAND_CASES)
print(f"\nPart A: {a_passed}/{len(EXPAND_CASES)}")

(3,)       -> (3,)       ref: ok     PASS
()         -> (2, 3)     ref: ok     PASS
(3,)       -> (2, 3)     ref: ok     PASS
(2, 1)     -> (2, 4)     ref: ok     PASS
(1, 3, 1)  -> (2, 3, 4)  ref: ok     PASS
(1,)       -> (0,)       ref: ok     FAIL
(0,)       -> (2, 0)     ref: ok     PASS
(2, 3)     -> (3,)       ref: error  PASS
(3,)       -> (4,)       ref: error  PASS
(3,)       -> (3, 1)     ref: error  PASS
(2, 1)     -> (1, 2)     ref: error  PASS
(0,)       -> (1,)       ref: error  PASS

Part A: 11/12


### D.2 — Parts B and C, hand-picked cases

8 broadcastable pairs, then 4 that are not. Part B is compared with `torch.broadcast_shapes`, part C with `torch.broadcast_tensors`.

In [8]:
BROADCAST_CASES = [
  ((3,),         (3,)),           # identical
  ((),           (2, 3)),         # scalar
  ((2, 1),       (1, 2)),         # both sides stretched
  ((5, 1, 4),    (3, 1)),         # different number of dimensions
  ((8, 1, 6, 1), (7, 1, 5)),      # NumPy docs example -> (8, 7, 6, 5)
  ((1,),         (0,)),           # 1 with 0 -> 0
  ((0, 1),       (1, 3)),         # size 0 with stretching -> (0, 3)
  ((1, 2, 3),    (2, 1, 1, 1)),   # -> (2, 1, 2, 3)
  ((0,),         (2,)),           # error
  ((2, 3),       (3, 2)),         # error
  ((4,),         (5,)),           # error
  ((2, 1, 3),    (4, 3, 1)),      # error in the first dimension
]


def test_broadcast(cases):
  passed_b = passed_c = 0
  for a_shape, b_shape in cases:
    A, B = rand_tensor(a_shape), rand_tensor(b_shape)

    # Part B vs torch.broadcast_shapes
    ref_b = outcome(lambda: torch.broadcast_shapes(A.shape, B.shape))
    mine_b = my_is_broadcastable(A, B)
    ok_b = (mine_b == (False, None)) if ref_b[0] == "error" else (mine_b == (True, ref_b[1]))

    # Part C vs torch.broadcast_tensors
    ok_c = agree(outcome(lambda: my_broadcast_tensors(A, B)),
                 outcome(lambda: torch.broadcast_tensors(A, B)))

    passed_b += ok_b
    passed_c += ok_c
    ref_str = str(tuple(ref_b[1])) if ref_b[0] == "ok" else "error"
    print(f"{str(a_shape):<14}{str(b_shape):<14} ref: {ref_str:<14} "
          f"B: {'PASS' if ok_b else 'FAIL'}  C: {'PASS' if ok_c else 'FAIL'}")
  return passed_b, passed_c

In [9]:
b_passed, c_passed = test_broadcast(BROADCAST_CASES)
print(f"\nPart B: {b_passed}/{len(BROADCAST_CASES)}   Part C: {c_passed}/{len(BROADCAST_CASES)}")

(3,)          (3,)           ref: (3,)           B: PASS  C: PASS
()            (2, 3)         ref: (2, 3)         B: PASS  C: PASS
(2, 1)        (1, 2)         ref: (2, 2)         B: PASS  C: PASS
(5, 1, 4)     (3, 1)         ref: (5, 3, 4)      B: PASS  C: PASS
(8, 1, 6, 1)  (7, 1, 5)      ref: (8, 7, 6, 5)   B: PASS  C: PASS
(1,)          (0,)           ref: (0,)           B: PASS  C: FAIL
(0, 1)        (1, 3)         ref: (0, 3)         B: PASS  C: FAIL
(1, 2, 3)     (2, 1, 1, 1)   ref: (2, 1, 2, 3)   B: PASS  C: PASS
(0,)          (2,)           ref: error          B: PASS  C: PASS
(2, 3)        (3, 2)         ref: error          B: PASS  C: PASS
(4,)          (5,)           ref: error          B: PASS  C: PASS
(2, 1, 3)     (4, 3, 1)      ref: error          B: PASS  C: PASS

Part B: 12/12   Part C: 10/12


### D.3 — Additional properties

Things a shape-and-value check doesn't cover: other dtypes, a non-contiguous input (a transposed tensor), that `B` isn't modified, and that gradients through our function match `expand_as` (relevant since these tensors are used in training).

In [10]:
def test_properties():
  results = []

  # dtype is preserved
  base = torch.randn(3, 1)
  for dt in [torch.bool, torch.int8, torch.float64, torch.complex64]:
    A = (base > 0) if dt == torch.bool else base.to(dt)
    B = torch.empty(2, 3, 4)
    results.append(same(my_expand_as(A, B), A.expand_as(B)))

  # non-contiguous input
  A = torch.arange(6).reshape(2, 3).t()    # shape (3, 2), not contiguous
  B = torch.empty(4, 3, 2)
  results.append(same(my_expand_as(A, B), A.expand_as(B)))

  # B is not modified
  A, B = rand_tensor((1, 4)), rand_tensor((3, 4))
  B_before = B.clone()
  my_expand_as(A, B)
  results.append(same(B, B_before))

  # gradients match expand_as (weighted sum, so each element gets a different gradient)
  W = torch.randn(3, 4)
  A1 = torch.randn(3, 1, requires_grad=True)
  A2 = A1.detach().clone().requires_grad_(True)
  (my_expand_as(A1, W) * W).sum().backward()
  (A2.expand_as(W) * W).sum().backward()
  results.append(torch.allclose(A1.grad, A2.grad))

  return results